# Minimum Spanning Forests and Kruskal


```{contents}
:local:
:depth: 2
```


```{index} Connect Every Site Without Paying for a Cycle
```



```{index} Keep Connectivity State Separate from the Cable Budget
```

## Keep Connectivity State Separate from the Cable Budget


A minimum spanning forest connects every pair of vertices that can be connected by the input, without paying for unnecessary cycles. This is different from a shortest route from one office. Kruskal processes edges by nondecreasing cost, but “choose cheap edges” is incomplete: an edge is accepted only if its endpoints are in different current components.

Union-find records those components. Each vertex starts as its own parent. `Find(v)` follows parent links to a root; equal roots mean already connected. `Union(a,b)` links the roots of two different components. Parent links here are bookkeeping, not the chosen cable edges. A parent link can point to an office with which there is no direct cable.

The small implementation below omits path compression so each parent change is easy to trace. Union by size attaches the smaller root's tree under the larger root. Sizes are meaningful at roots. The complete retained implementation adds path halving and graph validation after this walkthrough.


In [1]:
using System;
int[] parent = { 0, 1, 2, 3 }; // A, B, C, isolated D
int[] size = { 1, 1, 1, 1 };
int Find(int vertex)
{
    while (parent[vertex] != vertex) vertex = parent[vertex];
    return vertex;
}
bool Union(int a, int b)
{
    int ra = Find(a), rb = Find(b);
    if (ra == rb) return false;
    if (size[ra] < size[rb]) (ra, rb) = (rb, ra);
    parent[rb] = ra;
    size[ra] += size[rb];
    return true;
}
int cost = 0, components = 4;
foreach (var edge in new[] { (A: 0, B: 1, Cost: 1), (A: 1, B: 2, Cost: 2), (A: 0, B: 2, Cost: 5) })
{
    bool accepted = Union(edge.A, edge.B);
    if (accepted) { cost += edge.Cost; components--; }
    Console.WriteLine($"Cost {edge.Cost}: {(accepted ? "accept" : "reject")}");
}
Console.WriteLine($"Parents: {string.Join(",", parent)}");
Console.WriteLine($"Root A size: {size[Find(0)]}; total {cost}; components {components}");


Cost 1: accept
Cost 2: accept
Cost 5: reject
Parents: 0,0,0,3
Root A size: 3; total 3; components 2


The edge input is already sorted in this teaching cell; a general Kruskal implementation must sort arbitrary input first. A connects to B at cost one, then B connects to C at cost two. The cost-five edge connects vertices with the same root and is rejected. Isolated D remains a separate component rather than receiving a fictional cable.

| Edge | Roots before decision | Decision | Component sets afterward | Budget |
| --- | --- | --- | --- | ---: |
| A-B, 1 | A and B | Accept | {A,B}, {C}, {D} | 1 |
| B-C, 2 | A and C | Accept | {A,B,C}, {D} | 3 |
| A-C, 5 | A and A | Reject cycle | {A,B,C}, {D} | 3 |

After the second row, parent[C] is A even though the accepted edge was B-C. Find returns a component representative, not a route. To print the chosen network, retain accepted graph edges separately. Path compression changes parent links to speed future finds without changing connectivity or the cable budget.

Acyclicity establishes feasibility but not minimum cost. For a current component, consider the cut separating it from the rest of its input-connected component. A cheapest available crossing edge is safe. If an optimum omits that edge, add it to the optimum: the resulting cycle contains another crossing edge. Replacing that edge by the chosen edge does not raise cost. Thus the accepted choices can remain part of an optimum. Apply this reasoning within each input component when the graph is disconnected.


**Visualize it.** OpenDSA's [Kruskal's Algorithm](https://opendsa-server.cs.vt.edu/OpenDSA/Books/CS3/html/Kruskal.html) slideshow accepts and rejects edges in cost order, and [Union/Find](https://opendsa-server.cs.vt.edu/OpenDSA/Books/CS3/html/UnionFind.html) animates the parent-pointer forest that detects the cycles.

**Common mistake.** Reading a parent link as a chosen cable. After `Union(0, 2)`, vertex 2's parent may be 0 even though the accepted edge was between 1 and 2; the parent array answers "same component?", and the accepted-edge list answers "which cables?".


```{index} Guide a New Edge Before Running the Full Program
```

## Guide a New Edge Before Running the Full Program


Add C-D with cost four. Insert it at the correct place in the sorted edge order, and complete its roots, decision, component sets and budget. Then change A-C from five to zero and trace the entire order again. Do not keep the old acceptance decisions after changing costs.

```{dropdown} Check both variations
With C-D cost four, costs one, two and four are accepted, cost five is rejected, total cost is seven and the forest has one component. With A-C changed to zero as well, accept A-C at zero, A-B at one, reject B-C at two, and accept C-D at four: total five. Each connected four-vertex result has three edges.
```

For a tied-cost variation, explain why more than one edge set can be optimal. Verify declared vertices, acyclicity, component coverage, edge count and optimum cost as separate claims. The chapter assignments transfer these obligations to the facilities network; their independent-reference checks should follow this reasoning, not replace it.


```{index} Connect Every Site Without Paying for a Cycle
```

## Connect Every Site Without Paying for a Cycle



A facilities team wants cable connecting all sites at minimum total cable cost. This objective concerns the sum of selected edge costs, not the travel distance from headquarters. Start with three undirected links: A-B costs 2, B-C costs 2, and A-C costs 3. Selecting the two cost-2 links connects all three sites for four. Adding A-C would introduce a cycle and increase cost without connecting a new component.

Kruskal processes edges in nondecreasing weight. It accepts an edge when its endpoints are currently in different components. A disjoint-set structure records those components. Union does not mean join two vertex labels alphabetically; it combines the connectivity groups induced by already accepted edges.

| Candidate | Before | Decision | After |
| --- | --- | --- | --- |
| A-B, 2 | A / B / C | Accept | AB / C |
| B-C, 2 | AB / C | Accept | ABC |
| A-C, 3 | ABC | Reject cycle | ABC |

Acyclicity proves that selected edges form a forest, but not that it has minimum possible cost. The safe-choice argument uses a cut separating a current component from the rest. A cheapest crossing edge can replace an appropriate crossing edge in an optimal solution without making that solution more expensive. That exchange connects the local greedy choice to a global optimum. It differs from simply observing that the next edge is inexpensive.

Disconnected input has no spanning tree of all declared vertices. The meaningful extension is a minimum spanning forest, including isolated vertices in the component count. Negative edge weights do not break Kruskal's ordering/exchange reasoning, although the graph and edge contract must remain explicit. Equal weights can allow several correct forests; verify validity and optimum cost instead of requiring one exact edge list.

Guided case: add isolated D. Does the algorithm need an invented high-cost edge to connect D? No, if the contract promises a forest it reports another component. Now change A-C to cost one. Trace the accepted edges again and derive the new total before running the preserved implementation. Check edge count V minus component count, connectivity, and cost as different obligations.

For related explanation, see the further reading below.[^development-reading]


```{index} Spanning Trees and Shortest Paths
```

<span id="spanning-trees-and-shortest-paths"></span>


Facilities planning asks two different questions: which cables connect all offices at minimum total cost, and which routes minimize travel from HQ? Both use weights, but their objectives and graph models differ.

```{index} MST Contracts and Disconnected Graphs
```

## MST Contracts and Disconnected Graphs




```{index} MST Contracts and Disconnected Graphs
```

Kruskal uses undirected edges with explicit distinct vertices. Parallel edges are allowed; self-loops and unknown endpoints are rejected. Negative edge weights are valid for MSTs. If the graph is disconnected, return a minimum spanning forest and report component count; do not call it a spanning tree. An empty graph returns zero edges/cost/components.

In [ ]:
using System;
using System.Linq;
using System.Collections.Generic;

string[] vertices={"HQ","Lab","Shop","Depot","Remote"};
var edges=new[]{new GreedyGraph.Edge("HQ","Shop",2),new GreedyGraph.Edge("HQ","Lab",6),new GreedyGraph.Edge("Shop","Lab",3),new GreedyGraph.Edge("Lab","Depot",4),new GreedyGraph.Edge("Shop","Depot",5)};
var forest=GreedyGraph.Kruskal(vertices,edges,Console.WriteLine);
Console.WriteLine($"cost={forest.Cost}, edges={forest.Edges.Length}, components={forest.Components}");
public static class GreedyGraph
{
    public record Edge(string From,string To,int Weight);
    public record Forest(Edge[] Edges,long Cost,int Components);
    public sealed class Routes
    {
        public string Source { get; }
        public IReadOnlyDictionary<string,long?> Distance { get; }
        public IReadOnlyDictionary<string,string?> Previous { get; }
        internal Routes(string source,Dictionary<string,long?> distance,Dictionary<string,string?> previous)
        {
            Source=source;
            Distance=new System.Collections.ObjectModel.ReadOnlyDictionary<string,long?>(distance);
            Previous=new System.Collections.ObjectModel.ReadOnlyDictionary<string,string?>(previous);
        }
        public string[] PathTo(string target)
        {
            if(!Distance.ContainsKey(target))throw new KeyNotFoundException(target);
            if(Distance[target] is null)return Array.Empty<string>();
            var path=new Stack<string>();string? at=target;
            for(int steps=0;at is not null;steps++)
            {
                if(steps>=Distance.Count)throw new InvalidOperationException("Invalid predecessor chain.");
                path.Push(at);at=Previous[at];
            }
            if(path.Peek()!=Source)throw new InvalidOperationException("Path does not reach source.");
            return path.ToArray();
        }
    }
    // Undirected edges; parallel edges allowed. Negative MST weights are valid.
    public static Forest Kruskal(string[] vertices,Edge[] edges,Action<string>? trace=null)
    {
        Validate(vertices,edges,false);
        var parent=vertices.ToDictionary(v=>v,v=>v,StringComparer.Ordinal);
        var size=vertices.ToDictionary(v=>v,_=>1,StringComparer.Ordinal);
        int components=vertices.Length;long cost=0;var chosen=new List<Edge>();
        foreach(var edge in edges.OrderBy(e=>e.Weight).ThenBy(e=>e.From,StringComparer.Ordinal).ThenBy(e=>e.To,StringComparer.Ordinal))
        {
            string a=Find(edge.From),b=Find(edge.To);
            if(a==b){trace?.Invoke($"reject {edge.From}-{edge.To} ({edge.Weight}): cycle");continue;}
            if(size[a]<size[b])(a,b)=(b,a);
            parent[b]=a;size[a]+=size[b];components--;
            chosen.Add(edge);cost=checked(cost+edge.Weight);
            trace?.Invoke($"accept {edge.From}-{edge.To} ({edge.Weight}): components={components}");
        }
        return new Forest(chosen.ToArray(),cost,components);
        string Find(string vertex)
        {
            while(parent[vertex]!=vertex)
            {parent[vertex]=parent[parent[vertex]];vertex=parent[vertex];}
            return vertex;
        }
    }
    // Directed arcs; nonnegative weights required, including unreachable arcs.
    public static Routes Dijkstra(string[] vertices,Edge[] arcs,string source,Action<string>? trace=null)
    {
        Validate(vertices,arcs,true);
        var graph=vertices.ToDictionary(v=>v,_=>new List<Edge>(),StringComparer.Ordinal);
        if(!graph.ContainsKey(source))throw new KeyNotFoundException("Unknown source.");
        foreach(var edge in arcs)graph[edge.From].Add(edge);
        var distance=vertices.ToDictionary(v=>v,_=>(long?)null,StringComparer.Ordinal);
        var previous=vertices.ToDictionary(v=>v,_=>(string?)null,StringComparer.Ordinal);
        var queue=new PriorityQueue<string,long>();distance[source]=0;queue.Enqueue(source,0);
        while(queue.TryDequeue(out string? current,out long queued))
        {
            if(distance[current]!=queued){trace?.Invoke($"skip stale {current} ({queued})");continue;}
            trace?.Invoke($"settle {current} ({queued})");
            foreach(var edge in graph[current])
            {
                long candidate=checked(queued+edge.Weight);
                if(distance[edge.To] is null || candidate<distance[edge.To])
                {
                    distance[edge.To]=candidate;previous[edge.To]=current;
                    queue.Enqueue(edge.To,candidate);
                    trace?.Invoke($"relax {current}->{edge.To}: {candidate}");
                }
            }
        }
        return new Routes(source,distance,previous);
    }
    private static void Validate(string[] vertices,Edge[] edges,bool nonnegative)
    {
        ArgumentNullException.ThrowIfNull(vertices);ArgumentNullException.ThrowIfNull(edges);
        var known=new HashSet<string>(StringComparer.Ordinal);
        foreach(string vertex in vertices)
            if(vertex is null||!known.Add(vertex))throw new ArgumentException("Vertices must be distinct non-null labels.");
        foreach(var edge in edges)
        {
            if(edge is null||!known.Contains(edge.From)||!known.Contains(edge.To))throw new ArgumentException("Unknown endpoint.");
            if(edge.From==edge.To)throw new ArgumentException("This graph contract excludes self-loops.");
            if(nonnegative&&edge.Weight<0)throw new ArgumentOutOfRangeException(nameof(edges),"Dijkstra requires nonnegative weights.");
        }
    }
}


accept HQ-Shop (2): components=4
accept Shop-Lab (3): components=3
accept Lab-Depot (4): components=2
reject Shop-Depot (5): cycle
reject HQ-Lab (6): cycle
cost=9, edges=3, components=2


Here Remote is isolated, so the result has two components and three accepted edges. In general a forest on V vertices and c components has V−c edges. A nonempty connected result has V−1 edges. With tied weights, multiple optimal forests can exist; compare costs and validity rather than requiring one exact edge set.

**Common mistake.** Checking `edges == vertices - 1` to decide whether Kruskal succeeded. With `Remote` isolated the forest has three edges for five vertices and is still the correct answer; the check must use the component count, `V - c`.


```{index} Kruskal Invariants and the Cut Argument
```

## Kruskal Invariants and the Cut Argument




```{index} Kruskal Invariants and the Cut Argument
```

Accepted edges are acyclic. Union-find tracks exactly the components connected by accepted edges; an edge with equal roots would create a cycle. A minimum-cost crossing edge is safe: take the cut separating one current component from the rest. An optimal spanning forest that lacks the chosen edge can exchange a crossing edge on the resulting cycle for it without raising cost.

Within each connected input component, this cut argument shows the accepted forest can be extended to an optimum. Acyclicity alone would not prove minimality. The implementation uses union by size and path halving. Sorting costs O(E log E); disjoint-set work is amortized O((V+E) α(V)) under the standard union-find bound. Include O(V+E) setup/output storage. For parallel-edge graphs, avoid replacing log E with log V without an assumption relating them.

```{index} Inspect the Cable Forest
```

## Inspect the Cable Forest




```{index} Inspect the Cable Forest
```

Trace accepted and rejected edges, include an isolate, and verify edge count versus components. The total is a cable budget, not a travel-distance sum.

In [ ]:
using System;
using System.Linq;
using System.Collections.Generic;

string[] vertices={"HQ","Lab","Shop","Depot","Remote"};
var edges=new[]{new GreedyGraph.Edge("HQ","Shop",2),new GreedyGraph.Edge("HQ","Lab",6),new GreedyGraph.Edge("Shop","Lab",3),new GreedyGraph.Edge("Lab","Depot",4),new GreedyGraph.Edge("Shop","Depot",5)};
var forest=GreedyGraph.Kruskal(vertices,edges,Console.WriteLine);
if(forest.Edges.Length!=vertices.Length-forest.Components)throw new Exception("Forest count failed.");
Console.WriteLine($"cost={forest.Cost}, components={forest.Components}, spanning tree={forest.Components==1}");
public static class GreedyGraph
{
    public record Edge(string From,string To,int Weight);
    public record Forest(Edge[] Edges,long Cost,int Components);
    public sealed class Routes
    {
        public string Source { get; }
        public IReadOnlyDictionary<string,long?> Distance { get; }
        public IReadOnlyDictionary<string,string?> Previous { get; }
        internal Routes(string source,Dictionary<string,long?> distance,Dictionary<string,string?> previous)
        {
            Source=source;
            Distance=new System.Collections.ObjectModel.ReadOnlyDictionary<string,long?>(distance);
            Previous=new System.Collections.ObjectModel.ReadOnlyDictionary<string,string?>(previous);
        }
        public string[] PathTo(string target)
        {
            if(!Distance.ContainsKey(target))throw new KeyNotFoundException(target);
            if(Distance[target] is null)return Array.Empty<string>();
            var path=new Stack<string>();string? at=target;
            for(int steps=0;at is not null;steps++)
            {
                if(steps>=Distance.Count)throw new InvalidOperationException("Invalid predecessor chain.");
                path.Push(at);at=Previous[at];
            }
            if(path.Peek()!=Source)throw new InvalidOperationException("Path does not reach source.");
            return path.ToArray();
        }
    }
    // Undirected edges; parallel edges allowed. Negative MST weights are valid.
    public static Forest Kruskal(string[] vertices,Edge[] edges,Action<string>? trace=null)
    {
        Validate(vertices,edges,false);
        var parent=vertices.ToDictionary(v=>v,v=>v,StringComparer.Ordinal);
        var size=vertices.ToDictionary(v=>v,_=>1,StringComparer.Ordinal);
        int components=vertices.Length;long cost=0;var chosen=new List<Edge>();
        foreach(var edge in edges.OrderBy(e=>e.Weight).ThenBy(e=>e.From,StringComparer.Ordinal).ThenBy(e=>e.To,StringComparer.Ordinal))
        {
            string a=Find(edge.From),b=Find(edge.To);
            if(a==b){trace?.Invoke($"reject {edge.From}-{edge.To} ({edge.Weight}): cycle");continue;}
            if(size[a]<size[b])(a,b)=(b,a);
            parent[b]=a;size[a]+=size[b];components--;
            chosen.Add(edge);cost=checked(cost+edge.Weight);
            trace?.Invoke($"accept {edge.From}-{edge.To} ({edge.Weight}): components={components}");
        }
        return new Forest(chosen.ToArray(),cost,components);
        string Find(string vertex)
        {
            while(parent[vertex]!=vertex)
            {parent[vertex]=parent[parent[vertex]];vertex=parent[vertex];}
            return vertex;
        }
    }
    // Directed arcs; nonnegative weights required, including unreachable arcs.
    public static Routes Dijkstra(string[] vertices,Edge[] arcs,string source,Action<string>? trace=null)
    {
        Validate(vertices,arcs,true);
        var graph=vertices.ToDictionary(v=>v,_=>new List<Edge>(),StringComparer.Ordinal);
        if(!graph.ContainsKey(source))throw new KeyNotFoundException("Unknown source.");
        foreach(var edge in arcs)graph[edge.From].Add(edge);
        var distance=vertices.ToDictionary(v=>v,_=>(long?)null,StringComparer.Ordinal);
        var previous=vertices.ToDictionary(v=>v,_=>(string?)null,StringComparer.Ordinal);
        var queue=new PriorityQueue<string,long>();distance[source]=0;queue.Enqueue(source,0);
        while(queue.TryDequeue(out string? current,out long queued))
        {
            if(distance[current]!=queued){trace?.Invoke($"skip stale {current} ({queued})");continue;}
            trace?.Invoke($"settle {current} ({queued})");
            foreach(var edge in graph[current])
            {
                long candidate=checked(queued+edge.Weight);
                if(distance[edge.To] is null || candidate<distance[edge.To])
                {
                    distance[edge.To]=candidate;previous[edge.To]=current;
                    queue.Enqueue(edge.To,candidate);
                    trace?.Invoke($"relax {current}->{edge.To}: {candidate}");
                }
            }
        }
        return new Routes(source,distance,previous);
    }
    private static void Validate(string[] vertices,Edge[] edges,bool nonnegative)
    {
        ArgumentNullException.ThrowIfNull(vertices);ArgumentNullException.ThrowIfNull(edges);
        var known=new HashSet<string>(StringComparer.Ordinal);
        foreach(string vertex in vertices)
            if(vertex is null||!known.Add(vertex))throw new ArgumentException("Vertices must be distinct non-null labels.");
        foreach(var edge in edges)
        {
            if(edge is null||!known.Contains(edge.From)||!known.Contains(edge.To))throw new ArgumentException("Unknown endpoint.");
            if(edge.From==edge.To)throw new ArgumentException("This graph contract excludes self-loops.");
            if(nonnegative&&edge.Weight<0)throw new ArgumentOutOfRangeException(nameof(edges),"Dijkstra requires nonnegative weights.");
        }
    }
}


accept HQ-Shop (2): components=4
accept Shop-Lab (3): components=3
accept Lab-Depot (4): components=2
reject Shop-Depot (5): cycle
reject HQ-Lab (6): cycle
cost=9, components=2, spanning tree=False


The trace accepts HQ-Shop, Shop-Lab, and Lab-Depot while the components drop from 4 to 2, then rejects Shop-Depot and HQ-Lab as cycles, and ends with `cost=9, components=2, spanning tree=False`. `Remote` never appears in an edge, so it stays its own component, and three accepted edges for five vertices is exactly `V - c`.

**Common mistake.** Treating the forest's total cost as the cost of traveling between two offices. Four cables can connect everything cheaply while the route from HQ to Depot along them is longer than a direct cable the forest rejected; the shortest-path section makes this concrete.


```{rubric} Footnotes
```


[^development-reading]: Further reading: [Sedgewick and Wayne, Minimum Spanning Trees](https://algs4.cs.princeton.edu/43mst/index.php). The worked business scenarios and guided reasoning here are original teaching examples.
